# Do WER theo `vad.end_silence_ms` tren Kaggle

**Cau hoi:** engine ket thuc utterance (va reset ASR) sau 250 ms im lang. Neu cho ASR chay qua cac khoang ngung ngan
(400 / 600 / 800 ms) thi ASR co nhan dang tot hon khong (WER, so cau ASR rong), va doi lai bao nhieu do tre?
Y tuong lay tu bai bao X-Translator (muc 3.2): ASR chay lien tuc, lop chot segment cat doan de dich/hien thi.

**Truoc khi chay** (giong `kaggle_test.ipynb`)

1. Session Settings: Accelerator **GPU T4 x2**, **Internet ON**.
2. Add Data hai Kaggle Dataset:
   - `models`: noi dung `models/` (xem `scripts/copy-models.sh`).
   - `source`: zip cua repo khong co `models/`, `wheels_gpu/`, `.git` (lenh zip trong `docs/scripts-guide.md` muc 8).
     **Zip tu working tree hien tai** de co ban sua cat phan con lai bang doan tu (`segmenter.committed_end`).
3. Sua `MODELS_DATASET`, `SRC_DATASET`, `SILENCE_SWEEP` o cell cau hinh roi Run All. `LIMIT = 20` de chay thu nhanh.

**Chay gi:** `STEPS="B S"` cua `run_tests_kaggle.sh`: engine production (250 ms) chay buoc `B`
(`language_source=vi`, co dinh ngon ngu de tach rieng anh huong cua VAD), roi moi gia tri trong `SILENCE_SWEEP`
mot lan khoi dong engine voi `vad.end_silence_ms=<ms>` (nap lai model vai phut) va cung 200 clip.

**Doc ket qua**

- Bang WER: base `B_source_vi` (250 ms); `S<ms>_source_vi` = `end_silence_ms`. `delta_pp` am = tot hon;
  chenh duoi ~1 diem % la nhieu, nhin `ci95_pp` (khoang tin cay 95%) co chua 0 khong.
- Bang utterances (`logstats`): `utt_per_clip` (it hon = it bi bam cau), `empty_pct` (ASR tra rong),
  `max_utt_cuts` (bi cat cung o `vad.max_utterance_ms` 10 s), `utt_ms_p50/p90` (do dai utterance).
- `diff`: cac clip WER doi nhieu nhat, in ref / base / run de xem tan mat.

**Gioi han**

- Gui nhanh hon thoi gian thuc nen **khong do duoc do tre**. Do tre them la biet truoc: cau cuoi moi luot noi ra
  cham them `<ms> - 250` ms (VAD phai cho du khoang im lang moi chot cau).
- T4 khong phai L4; dataset la cau doc (DataStudio/Vietnamese_ASR_TestingData), it khoang ngung hon hoi thoai
  dien thoai, nen ket qua la chieu huong, khong phai con so production.


In [ ]:
import os, subprocess

MODELS_DATASET = "models"       # ten dataset trong /kaggle/input
SRC_DATASET = "source"          # ten dataset chua source (zip hoac thu muc da giai nen)

SILENCE_SWEEP = "400 600 800"   # cac gia tri vad.end_silence_ms (ms) de so voi production 250 ms
LIMIT = 0                       # 0 = du 200 clip; dat 20 de chay thu nhanh
CONCURRENCY = 4

ENGINE_DIR = "/kaggle/working/engine"
OUT_DIR = "/kaggle/working/out"
PY = "/kaggle/working/venv/bin/python"

def sh(cmd, check=True):
    print("$", cmd)
    proc = subprocess.run(["bash", "-lc", cmd], text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f"exit {proc.returncode}: {cmd}")

sh("ls /kaggle/input; nvidia-smi -L")

In [ ]:
# Tim thu muc dataset (Kaggle co the dat dataset duoi /kaggle/input/<ten> hoac /kaggle/input/datasets/<owner>/<ten>)
def find_dataset(name):
    hits = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        if os.path.basename(root) == name:
            hits.append(root)
        if root.count(os.sep) >= 5:
            dirs[:] = []
    if not hits:
        raise FileNotFoundError(f"khong thay dataset '{name}' trong /kaggle/input")
    return sorted(hits, key=len)[0]

models_dir = find_dataset(MODELS_DATASET)
src_dir = find_dataset(SRC_DATASET)
print("models:", models_dir)
print("source:", src_dir)

In [ ]:
# Chep source sang noi ghi duoc (giai nen neu la zip)
sh(f"rm -rf {ENGINE_DIR} && mkdir -p {ENGINE_DIR}")
zips = [f for f in os.listdir(src_dir) if f.endswith(".zip")]
if zips:
    sh(f"cd {ENGINE_DIR} && unzip -q '{src_dir}/{zips[0]}'")
else:
    sh(f"cp -a '{src_dir}/.' {ENGINE_DIR}/")

# neu zip co thu muc goc duy nhat thi nhay vao trong
entries = os.listdir(ENGINE_DIR)
if "scripts" not in entries and len(entries) == 1:
    ENGINE_DIR = os.path.join(ENGINE_DIR, entries[0])
assert os.path.exists(f"{ENGINE_DIR}/scripts/kaggle/setup_kaggle.sh"), "khong thay scripts/kaggle trong source"
# source phai co buoc S va ban sua doan tu; zip cu thi dung lai som
assert "SILENCE_SWEEP" in open(f"{ENGINE_DIR}/scripts/kaggle/run_tests_kaggle.sh", encoding="utf-8").read(), \
    "run_tests_kaggle.sh trong source chua co buoc S: zip lai tu working tree hien tai"
assert "def committed_end" in open(f"{ENGINE_DIR}/src/nemotron_testbed/segmenter.py", encoding="utf-8").read(), \
    "segmenter.py trong source chua co committed_end: zip lai tu working tree hien tai"
sh(f"ls {ENGINE_DIR}")

In [ ]:
# Cai dat (vai chuc phut lan dau): venv Python 3.12, torch cu124, lock, piper-tts, patch NeMo
sh(f"cd {ENGINE_DIR} && MODELS_DIR='{models_dir}' bash scripts/kaggle/setup_kaggle.sh")

In [ ]:
# B (production 250 ms) roi tung gia tri trong SILENCE_SWEEP; moi gia tri mot lan khoi dong engine (vai phut)
env = f"STEPS='B S' SILENCE_SWEEP='{SILENCE_SWEEP}' LIMIT={LIMIT} CONCURRENCY={CONCURRENCY}"
sh(f"cd {ENGINE_DIR} && {env} bash scripts/kaggle/run_tests_kaggle.sh")

In [ ]:
# Ket qua tong hop: bang WER (base = B_source_vi, 250 ms) va bang utterances theo tung lan chay
print(open(f"{OUT_DIR}/summary.txt", encoding="utf-8").read())

In [ ]:
# Clip WER doi nhieu nhat so voi 250 ms (xau hon / tot hon), in ref / base / run
for ms in SILENCE_SWEEP.split():
    print("=" * 30, f"end_silence_ms {ms} vs 250")
    sh(f"{PY} {ENGINE_DIR}/scripts/kaggle/eval_wer.py diff --base {OUT_DIR}/B_source_vi.jsonl "
       f"{OUT_DIR}/S{ms}_source_vi.jsonl -n 5", check=False)

In [ ]:
# Kiem tra: moi engine chay dung end_silence_ms (main = gia tri cua etc/nemotron-engine.yaml), va log co du
sh(f"grep -H 'end_silence_ms' {OUT_DIR}/config-*.yaml", check=False)
sh(f"ls -la {OUT_DIR} {OUT_DIR}/logs/*", check=False)